# Hidden Markov regimes: filtered versus smoothed

This notebook uses a reproducible synthetic return series with two hidden volatility states. It is a controlled learning exercise, not a market estimate. The model fits only the early chronological sample; filtered probabilities then process returns one by one.

Read [`docs/regime-switching.md`](../docs/regime-switching.md) before treating a probability plot as a signal.

## 1. Simulate a hidden two-state process

The states are deliberately persistent, with different return volatilities. Real data do not reveal their true hidden state.

In [ ]:
import numpy as np
import pandas as pd

from quant_models.regimes.gaussian_hmm import fit_gaussian_regime_model

random = np.random.default_rng(42)
n_observations = 1_000
true_states = np.zeros(n_observations, dtype=int)
for row in range(1, n_observations):
    switch_probability = 0.025
    true_states[row] = (
        1 - true_states[row - 1]
        if random.random() < switch_probability
        else true_states[row - 1]
    )
returns = random.normal(
    loc=np.array([0.0002, -0.0005])[true_states],
    scale=np.array([0.006, 0.028])[true_states],
)
dates = pd.bdate_range("2020-01-01", periods=n_observations)
synthetic_returns = pd.Series(returns, index=dates, name="daily_log_return")
pd.DataFrame(
    {"simulated_return": synthetic_returns, "hidden_state": true_states}, index=dates
).head()

## 2. Fit on the beginning, filter forward

The split is chronological, not random. Filtered probabilities only use returns observed through each date. Smoothed probabilities use the full series and therefore revise the past using future data.

In [ ]:
result = fit_gaussian_regime_model(
    synthetic_returns,
    n_states=2,
    train_fraction=0.7,
    n_restarts=5,
    max_iterations=200,
    random_seed=42,
)
print(
    f"Converged: {result.converged}; likelihood decreases in "
    f"{result.restarts_with_likelihood_decreases} restart(s)"
)
result.daily_volatilities, result.transition_matrix

In [ ]:
ax = result.filtered_probabilities.plot(
    title="Filtered state probabilities (synthetic data)",
    ylabel="Probability using data through that date",
    ylim=(0, 1),
    figsize=(12, 4),
)
ax.axvline(result.training_end, color="black", linestyle="--", label="Training cutoff")
ax.legend()

## 3. Compare smoothing carefully

A smoothed probability can be visually cleaner because future observations help reconstruct the state at a past date. That is exactly why it would create look-ahead bias in a backtest.


In [ ]:
comparison = pd.concat(
    {
        "Filtered": result.filtered_probabilities.iloc[:, 1],
        "Smoothed (future data)": result.smoothed_probabilities.iloc[:, 1],
    },
    axis=1,
)
comparison.plot(
    title="Filtered and smoothed high-volatility probabilities",
    ylabel="Probability",
    ylim=(0, 1),
    figsize=(12, 4),
)

## Questions

1. Did the model recover two visibly different volatility levels?
2. How do filtered and smoothed probabilities differ around a state change?
3. Why is the higher-volatility state not necessarily a crash signal?
4. What might change if you alter the training fraction, state count, or random seed?
